# DDM spread-feedback divergence pre-check
Throwaway experiment: does `colab_runner.py`'s generic exec_notebook/download path work for a plain diagnostic script (not a TrainingRun)? Checks which (delta, wma) candidates diverge under spread_feedback_a=1.0, window=150.

In [ ]:
import random
import numpy as np
import json
import time

In [ ]:
# --- Embedded from backend/data/collectors/ddm_simulator.py (inspect.getsource) ---
from collections.abc import Iterable
import pandas as pd

class _DDMv1:
    """DeterministicDealerModelV1 (Takayasu et al.).

    Closely mirrors the original DeterministicDealerModelV1:
      - Agent positions, tendencies, and prices are numpy arrays (faster than
        the original pandas DataFrame, but otherwise identical logic).
      - Time advances via _add_time_for_ticks() on every step, exactly as in
        the original's simulate(total_seconds) loop.
      - Balanced 50/50 position initialisation (matches original bull_ratio=0.5).
    """

    def __init__(
        self,
        num_agent: int,
        max_volatility: float = 0.02,
        min_volatility: float = 0.01,
        trade_unit: float = 0.001,
        initial_price: float = 100.0,
        spread: float = 1.0,
        tick_time: float = 0.001,
        time_noise_method: str | None = "exp",
        max_noise_factor: int = 100,
        initial_positions: list | None = None,
        **kwargs,
    ) -> None:
        # Tendency values — rounded to the same precision as trade_unit,
        # matching the original's `tendency.round(decimal_num)` logic.
        tend_raw = [random.uniform(min_volatility, max_volatility) for _ in range(num_agent)]
        if trade_unit < 1:
            decimal_num = 0
            d = 0.1
            for _ in range(100):
                decimal_num += 1
                if trade_unit / d >= 1 - trade_unit:
                    break
                d *= 0.1
            self.tend = np.round(np.array(tend_raw, dtype=np.float64), decimal_num)
        else:
            self.tend = np.array(tend_raw, dtype=np.float64)

        self.agent_prices = np.array(
            [random.uniform(initial_price, initial_price + spread) for _ in range(num_agent)],
            dtype=np.float64,
        )

        if initial_positions is not None:
            self.position = np.array(initial_positions, dtype=np.int8)
        else:
            # Balanced 50/50 — matches original bull_ratio=0.5.
            n_long = num_agent // 2
            pos: list[int] = [1] * n_long + [-1] * (num_agent - n_long)
            random.shuffle(pos)
            self.position = np.array(pos, dtype=np.int8)

        self.spread = spread
        self.market_price: float = initial_price + spread
        self.trade_unit = trade_unit
        self.tick_time: float = 0.0
        self.tick_time_unit: float = tick_time
        self.price_history: list[float] = [self.market_price]

        # Time-noise function — mirrors original's __get_time_noise setup.
        if time_noise_method is None or time_noise_method == "uniform":
            self._max_noise_factor = max_noise_factor
            self.__get_time_noise = self._get_uniform_noise
        elif time_noise_method == "exp":
            self.__noise_factors = list(range(1, max_noise_factor + 1))
            self.__noise_weights = [1 / (i + 1) for i in range(max_noise_factor)]
            self.__get_time_noise = self._get_weighted_noise
        else:
            raise ValueError(f"Unknown time_noise_method: {time_noise_method!r}")

    # -- noise helpers (mirror original) ------------------------------------

    def _get_uniform_noise(self) -> float:
        return random.uniform(1, self._max_noise_factor)

    def _get_weighted_noise(self) -> float:
        return random.choices(self.__noise_factors, weights=self.__noise_weights)[0]

    # -- core simulation steps (mirror original) ----------------------------

    def advance_order_price(self) -> None:
        """Advance all agent prices by position × tendency (no-trade step)."""
        self.agent_prices += self.position * self.tend

    def _add_time_for_ticks(self) -> None:
        """Advance simulated time by tick_time_unit × noise (every step)."""
        self.tick_time += self.tick_time_unit * self.__get_time_noise()

    def _contruct(self) -> bool:
        """Attempt to match a trade. Returns True if a trade occurred."""
        long_mask = self.position == 1
        short_mask = self.position == -1
        if not long_mask.any() or not short_mask.any():
            return False

        long_prices = self.agent_prices[long_mask]
        short_prices = self.agent_prices[short_mask]

        ask_price = long_prices.max()
        bid_price = short_prices.min() + self.spread

        if ask_price >= bid_price:
            self.market_price = (
                (ask_price + bid_price) / 2 // self.trade_unit
            ) * self.trade_unit
            long_idx = np.where(long_mask)[0][np.argmax(long_prices)]
            short_idx = np.where(short_mask)[0][np.argmin(short_prices)]
            self.position[long_idx] = -1
            self.position[short_idx] = 1
            return True
        return False

    def _common_step(self) -> tuple[float | None, float]:
        """One simulation step: try to trade, else advance prices."""
        if self._contruct():
            self.price_history.append(self.market_price)
            return self.market_price, self.tick_time
        else:
            self.advance_order_price()
            return None, self.tick_time

    def simulate_stream(
        self,
        n_trades: int | None = None,
        total_seconds: float | None = None,
        yield_interval: int = 500,
    ):
        """Yield (price, tick_time) for each trade.

        Mirrors the original simulate(total_seconds) loop:
          - _add_time_for_ticks() is called on EVERY step (trade and no-trade).
          - The trade timestamp is self.tick_time at the moment of the trade.

        Stop conditions (first one that triggers wins):
          - total_seconds: stop when self.tick_time >= total_seconds
          - n_trades: stop after exactly n_trades yields
          - Neither: run forever (endless mode)

        yield_interval: call time.sleep(0) every this many steps to release the
          GIL and let other threads (web server, scheduler) get CPU time.

        Raises RuntimeError if no trade occurs for 10M consecutive steps
        (indicates model divergence — only a risk with num_agent < 50).
        """
        import itertools
        import time

        # Retention must cover whichever lookback window is actually in use -- DDMv3's own WMA,
        # and (when enabled) the stress-dependent spread's own moving-average window, which the
        # book sets much longer (100-200) than a typical wma. Using only `wma` here would
        # silently truncate price_history and corrupt _update_spread()'s moving average.
        _spread_window = getattr(self, "spread_feedback_window", 0) if getattr(self, "spread_feedback_a", None) is not None else 0
        keep = max(getattr(self, "wma", 0), _spread_window) + 2
        yielded = 0
        no_trade_streak = 0
        step = 0
        _MAX_NO_TRADE = 10_000_000

        for _ in itertools.count():
            self._add_time_for_ticks()

            # Yield the GIL periodically so web-server and other threads stay
            # responsive even while the simulation is burning a full core.
            step += 1
            if step % yield_interval == 0:
                time.sleep(0)

            if total_seconds is not None and self.tick_time >= total_seconds:
                return

            price, tick = self._common_step()
            if price is not None:
                if len(self.price_history) > keep:
                    del self.price_history[0]
                yield price, tick
                yielded += 1
                no_trade_streak = 0
                if n_trades is not None and yielded >= n_trades:
                    return
            else:
                no_trade_streak += 1
                if no_trade_streak >= _MAX_NO_TRADE:
                    raise RuntimeError(
                        f"DDM simulation stalled: no trade in {_MAX_NO_TRADE:,} steps. "
                        "Agent prices likely diverged — use num_agent ≥ 300."
                    )

    def simulate(self, n_trades: int) -> pd.DataFrame:
        """Convenience: run for n_trades and return a DataFrame of prices."""
        prices = [p for p, _ in self.simulate_stream(n_trades=n_trades)]
        return pd.DataFrame({"price": prices})


class DDMv3(_DDMv1):
    """DeterministicDealerModelV3 — adds WMA trend-following feedback."""

    def __init__(
        self,
        num_agent: int,
        max_volatility: float = 0.02,
        min_volatility: float = 0.01,
        trade_unit: float = 0.001,
        initial_price: float = 100.0,
        spread: float = 1.0,
        tick_time: float = 0.001,
        time_noise_method: str | None = "exp",
        max_noise_factor: int = 100,
        dealer_sensitive: float | Iterable | None = None,
        wma: int | Iterable = 5,
        dealer_sensitive_min: float = -3.5,
        dealer_sensitive_max: float = -1.5,
        spread_feedback_a: float | None = None,
        spread_feedback_window: int = 150,
        **kwargs,
    ) -> None:
        super().__init__(
            num_agent, max_volatility, min_volatility, trade_unit,
            initial_price, spread, tick_time, time_noise_method, max_noise_factor,
        )
        self.price_history = [self.market_price]

        # spread_feedback_a (opt-in, default None preserves existing behaviour exactly): a
        # stress-dependent spread, z(n) = z(0) + a*|P(n) - P_m(n)|, per 高安・和泉・山田・水田
        # 『マルチエージェントによる金融市場のシミュレーション』(コロナ社) eq. 4.28-4.33. z(0) is
        # this constructor's own `spread` argument. The SAME z(n) is used both as the matching
        # threshold in _contruct() (inherited, unchanged) and to amplify the random price-noise
        # term in advance_order_price() below -- the book's own equations use one shared z(n) for
        # both purposes (no per-agent index on z(n) in eq. 4.31), so this isn't a conflation of
        # two different quantities, confirmed against the source before implementing. It does
        # introduce a feedback loop absent from the base model (wider spread -> harder to match
        # AND noisier prices -> potentially wider spread next step) -- expected to possibly
        # destabilize some (delta, wma) combinations that were stable without it; that is a
        # property to characterize, not a bug to suppress.
        self.spread_feedback_a = spread_feedback_a
        self.spread_feedback_window = spread_feedback_window
        self._spread_z0 = spread

        if dealer_sensitive is None:
            self.dealer_sensitive = np.array(
                [random.uniform(dealer_sensitive_min, dealer_sensitive_max) for _ in range(num_agent)]
            )
        elif isinstance(dealer_sensitive, (int, float)):
            self.dealer_sensitive = np.full(num_agent, float(dealer_sensitive))
        else:
            self.dealer_sensitive = np.asarray(dealer_sensitive)

        if isinstance(wma, int):
            self.weight_array = np.array([random.uniform(0, 1) for _ in range(wma)])
            self.wma = wma
        else:
            self.weight_array = np.asarray(wma)
            self.wma = len(self.weight_array)
        self._total_weight = self.weight_array.sum()

    def _wma_diff(self) -> float:
        if len(self.price_history) < self.wma + 1:
            return 0.0
        h = self.price_history
        diffs = [h[-i] - h[-i - 1] for i in range(1, self.wma + 1)]
        return float(np.dot(self.weight_array, diffs) / self._total_weight)

    def _update_spread(self) -> None:
        """z(n) = z(0) + a*|P(n) - P_m(n)| (eq. 4.31). No-op when the feedback is disabled, which
        leaves self.spread permanently equal to _spread_z0 -- the noise_amp ratio in
        advance_order_price() below is then exactly 1.0, reproducing the base model exactly."""
        if self.spread_feedback_a is None:
            return
        window = self.price_history[-self.spread_feedback_window :]
        if len(window) < 2:
            return
        p_now = self.price_history[-1]
        p_ma = sum(window) / len(window)
        self.spread = self._spread_z0 + self.spread_feedback_a * abs(p_now - p_ma)

    def _common_step(self) -> tuple[float | None, float]:
        """Same as _DDMv1._common_step, but updates the (possibly time-varying) spread BEFORE
        the matching attempt, so _contruct() always sees the current z(n)."""
        self._update_spread()
        return super()._common_step()

    def advance_order_price(self) -> None:
        """Advance agent prices with WMA trend-following feedback, and (when spread feedback is
        enabled) an amplified random-noise term: (z(n)/z(0)) scales position*tend, mirroring eq.
        4.33's (z(n)/z(0))*f_i(t) -- the trend-following term `follow` is intentionally NOT
        scaled, matching the book's own equation."""
        wma = self._wma_diff()
        follow = self.dealer_sensitive * wma
        noise_amp = self.spread / self._spread_z0
        self.agent_prices += self.position * self.tend * noise_amp + follow


In [ ]:

CANDIDATES = [(-1.5, 15), (-1.4, 15), (-1.3, 15), (-1.2, 15), (-1.1, 15), (-1.0, 10), (-0.9, 10), (-0.8, 7), (-0.7, 7), (-0.6, 7), (-0.5, 7), (-0.4, 5), (-0.3, 5), (-0.2, 10), (-0.1, 7), (0.0, 15), (-0.8, 5), (-0.5, 10), (0.0, 20), (-1.0, 20)]
SPREAD_A = 1.0
SPREAD_WINDOW = 150
N_TRADES = 28800

results = []
t0 = time.time()
for delta, wma in CANDIDATES:
    random.seed(0); np.random.seed(0)
    model = DDMv3(num_agent=300, max_volatility=0.02, min_volatility=0.01, wma=wma,
                  dealer_sensitive_min=-3.5+delta, dealer_sensitive_max=-1.5+delta,
                  spread_feedback_a=SPREAD_A, spread_feedback_window=SPREAD_WINDOW)
    status = "ok"
    amps = []
    try:
        for price, tt in model.simulate_stream(n_trades=N_TRADES):
            amps.append(model.spread / model._spread_z0)
    except RuntimeError:
        status = "diverged"
    except (ValueError, OverflowError):
        status = "error"
    entry = {"delta": delta, "wma": wma, "status": status}
    if amps:
        arr = np.array(amps)
        if np.all(np.isfinite(arr)):
            entry.update(mean_amp=float(arr.mean()), p99_amp=float(np.percentile(arr, 99)), max_amp=float(arr.max()))
        else:
            entry["status"] = "non_finite"
    results.append(entry)
    print(entry)

print(f"done in {time.time()-t0:.1f}s")
with open("/content/divergence_check.json", "w") as f:
    json.dump(results, f, indent=2)
print("wrote /content/divergence_check.json")
